# 第 2 節｜BERT 二元分類微調

這次只做**一個二元分類任務**：判斷留言是否表示「已確認參加校園園遊會」。不再進行三元分類或第二次模型訓練。

**資料標籤定義（請先看）：**

| `label` | 代表意義 | 留言例子 |
|---|---|---|
| `1` | 已確認會參加 | 我當天會到場幫忙。 |
| `0` | 尚未確認會參加（包含確定無法參加、尚未決定） | 那天不能到。／我還不確定。 |

這不是「一定不參加 vs 一定參加」的分類：**標籤 0 只表示沒有確認會出席**。500 筆留言都是模擬資料，這次只把原三類標籤整合為兩類，沒有更動留言內容。

**學習方式：**5 組「Markdown 範例 → 空白練習 Cell」；Tokenizer、Trainer 的較複雜設定已寫好，請直接執行。建議 Colab 選 **T4 GPU**，首次會下載 BERT 權重，實際訓練時間依環境而異。


## 準備環境｜直接執行，不需抄寫

請先將 **`03_校園園遊會留言_500筆_二元分類.csv`** 上傳到 Colab 左側檔案區（通常位於 `/content`），再開始練習。


In [ ]:
# Install the libraries required by this notebook.
# 安裝 Notebook 所需套件。
%pip -q install transformers==4.57.1 datasets==3.6.0 accelerate sentencepiece


In [ ]:
# Import the tools used throughout this notebook.
# 集中匯入整份 Notebook 需要的套件。
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline


## 一、讀取資料與切分資料

每列是一句留言；`text` 是輸入 X，`label` 是正確答案 y。`message_id` 只是編號，不是模型輸入。


### 練習 1｜讀取 500 筆留言

確認 CSV 已放到 Colab 的檔案區。

```python
# Read the binary classification dataset.
# 讀取二元分類的園遊會留言資料。
df = pd.read_csv("03_校園園遊會留言_500筆_二元分類.csv")

# Preview the first five examples.
# 查看前五筆留言和標籤。
display(df.head())
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 2｜切分訓練集與測試集

訓練集讓模型學習；測試集先保留，訓練後才用來評估。`stratify` 讓兩組資料維持相近的 0／1 比例。

```python
# Split 80 percent for training and 20 percent for testing.
# 將 80% 資料用來訓練，20% 留作測試。
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Show the dataset sizes.
# 查看訓練集與測試集筆數。
print("訓練：", len(train_df), "測試：", len(test_df))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 二、建立 BERT 訓練工具｜下面三格直接執行

這裡有三個工具：

1. `Tokenizer`：把文字轉成模型能讀取的數字。
2. `Model`：使用預訓練中文 BERT，將分類輸出設定為 **2 類**。
3. `Trainer`：負責執行訓練。

這幾格屬於進階設定，**不要求學生自行輸入或理解每個參數**。本課先訓練 1 個 Epoch；模型在訓練階段不使用測試集。


In [ ]:
# Load the matching tokenizer for Chinese BERT.
# 載入中文 BERT 使用的文字轉換工具。
MODEL_ID = "google-bert/bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Convert training sentences into numerical tokens.
# 將訓練留言轉成模型使用的數字。
train_tokens = tokenizer(train_df["text"].tolist(), truncation=True, padding="max_length", max_length=64)

# Convert test sentences using the same tokenizer.
# 使用相同工具轉換測試留言。
test_tokens = tokenizer(test_df["text"].tolist(), truncation=True, padding="max_length", max_length=64)

# Attach the training answers to the numerical inputs.
# 將訓練資料的正確答案加入模型輸入。
train_tokens["labels"] = train_df["label"].tolist()

# Attach the test answers to the numerical inputs.
# 將測試資料的正確答案加入模型輸入。
test_tokens["labels"] = test_df["label"].tolist()

# Prepare both groups in the format expected by Trainer.
# 轉成 Trainer 可以使用的資料格式。
train_data = Dataset.from_dict(train_tokens)
test_data = Dataset.from_dict(test_tokens)

print("訓練筆數：", len(train_data), "測試筆數：", len(test_data))


In [ ]:
# Load pretrained BERT with a two-class output layer.
# 載入預訓練 BERT，並設定兩個分類結果。
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: "尚未確認參加", 1: "確定參加"},
    label2id={"尚未確認參加": 0, "確定參加": 1}
)

# The classification layer is newly created and needs training.
# 新建立的分類層尚未學習，稍後會透過微調進行訓練。
print("類別數：", model.config.num_labels)


In [ ]:
# Set only the essential training options.
# 設定少量必要的訓練參數。
settings = TrainingArguments(
    output_dir="03_bert_binary_output",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=3e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

# Connect the model, settings, and training data.
# 將模型、訓練參數與訓練資料交給 Trainer。
trainer = Trainer(model=model, args=settings, train_dataset=train_data)


## 三、開始微調、預測與評估

接下來只有三組短練習：**訓練 → 測試 → 輸入一句新留言**。


### 練習 3｜開始訓練 BERT

只要執行 `trainer.train()`；T4 GPU 可能需要幾分鐘。這一步真的會更新模型權重。

```python
# Fine-tune BERT using the training examples.
# 使用訓練資料微調 BERT。
trainer.train()
```


In [ ]:
# 請在此處根據範例練習程式碼


### 練習 4｜計算測試集 Accuracy

`predict()` 產生模型預測；`argmax()` 取分數最高的類別；Accuracy 是預測正確的比例。

```python
# Predict labels for the untouched test set.
# 預測先前保留的測試資料。
test_result = trainer.predict(test_data)

# Choose the class with the highest prediction score.
# 選擇分數最高的分類結果。
predicted = test_result.predictions.argmax(axis=1)

# Calculate the classification accuracy.
# 計算分類正確率。
print("Accuracy:", round(accuracy_score(test_df["label"], predicted), 3))
```


In [ ]:
# 請在此處根據範例練習程式碼


## 準備新留言預測｜直接執行這格

將剛剛微調好的模型轉成簡單的 `pipeline()`，下一格學生只需要修改一句文字。


In [ ]:
# Use the fine-tuned model to create a text classifier.
# 使用剛微調的模型建立文字分類工具。
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, device=device)


### 練習 5｜自己寫一句話，讓模型分類

直接修改 `text`，例如「我那天要補習，無法到場。」或「我可能可以去，但還在等家人回覆。」

```python
# Write one new message.
# 輸入一句新的園遊會留言。
text = "我已經確定可以參加園遊會！"

# Predict whether attendance is confirmed.
# 預測這句話是否表示已確認參加。
result = classifier(text)[0]

# Display the label and model score.
# 顯示中文分類結果及分數。
print("分類：", result["label"])
print("模型分數：", round(result["score"], 3))
```


In [ ]:
# 請在此處根據範例練習程式碼


## Optional｜查看訓練 Loss（直接執行）

Loss 越低通常表示訓練資料上的預測誤差越小，但**不代表測試資料一定更準**。


In [ ]:
# Convert the model training records to a table.
# 將模型訓練紀錄整理成表格。
history = pd.DataFrame(trainer.state.log_history)

# Draw the recorded training loss.
# 繪製訓練過程中的 Loss 曲線。
history.plot(x="step", y="loss", marker="o", title="BERT Training Loss")

# Display the chart.
# 顯示圖表。
plt.show()


## Optional｜查看模型預測的 0／1 人數（直接執行）

這裡不是再訓練一次，而是檢視前面 `predict()` 的輸出。


In [ ]:
# Count predicted labels in the test set.
# 統計測試資料中各個預測類別的數量。
print("預測為 0／1 的筆數：")
print(pd.Series(predicted).value_counts().sort_index())

# Show the real label counts for comparison.
# 顯示真實標籤的筆數，方便比較。
print("真實的 0／1 筆數：")
print(test_df["label"].value_counts().sort_index())

# Compare with a classifier that always predicts zero.
# 比較「全部猜 0」的簡單基準正確率。
baseline = (test_df["label"] == 0).mean()
print("全部猜 0 的 Accuracy：", round(baseline, 3))


## Optional｜儲存微調模型（直接執行）

如果想在之後重新載入模型，可以儲存到 Colab 的資料夾。Colab 執行階段重設後，本機暫存檔會消失；重要檔案請自行另外備份。


In [ ]:
# Save the fine-tuned BERT model.
# 儲存已微調的 BERT 模型。
trainer.save_model("03_bert_binary_model")

# Save the tokenizer used by this model.
# 儲存模型使用的文字轉換工具。
tokenizer.save_pretrained("03_bert_binary_model")

print("已儲存模型資料夾：03_bert_binary_model")


**課堂總結：**讀取留言 → 切分資料 → 轉換文字 → 微調 BERT → 預測測試集 → 計算 Accuracy → 測試新留言。本教材只有二元分類，**不會出現 `LABEL_2`**；微調模型已設定中文類別名稱。這份資料有相似的模擬句型，成績可能較樂觀；如需反覆調參，應另外建立驗證集，避免用測試集選參數。
